# Exam 16th of January 2026, 8.00-13.00 for the course 1MS041 (Introduction to Data Science / Introduktion till dataanalys)

## Instructions:
1. Complete the problems by following instructions.
2. When done, submit this file with your solutions saved, following the instruction sheet.

This exam has 3 problems for a total of 40 points, to pass you need
20 points. The bonus will be added to the score of the exam and rounded afterwards.

## Some general hints and information:
* Try to answer all questions even if you are uncertain.
* Comment your code, so that if you get the wrong answer I can understand how you thought
this can give you some points even though the code does not run.
* Follow the instruction sheet rigorously.
* This exam is partially autograded, but your code and your free text answers are manually graded anonymously.
* If there are any questions, please ask the exam guards, they will escalate it to me if necessary.

## Tips for free text answers
* Be VERY clear with your reasoning, there should be zero ambiguity in what you are referring to.
* If you want to include math, you can write LaTeX in the Markdown cells, for instance `$f(x)=x^2$` will be rendered as $f(x)=x^2$ and `$$f(x) = x^2$$` will become an equation line, as follows
$$f(x) = x^2$$
Another example is `$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$` which renders as
$$f_{Y \mid X}(y,x) = P(Y = y \mid X = x) = \exp(\alpha \cdot x + \beta)$$

## Finally some rules:
* You may not communicate with others during the exam, for example:
    * You cannot ask for help in Stack-Overflow or other such help forums during the Exam.
    * You may not communicate with AI's, for instance ChatGPT.
    * Your on-line and off-line activity is being monitored according to the examination rules.

## Good luck!

In [ ]:
# Insert your anonymous exam ID as a string in the variable below
examID="0069-AGZ"


---
## Exam vB, PROBLEM 1
Maximum Points = 14


This problem is about **SVD** and a simple **anomaly detection** idea using low-rank reconstruction.



Unless stated otherwise, when you are asked to produce a matrix or vector, it must be a **NumPy array**.

1. **[4p] SVD.** Load `data/SVD.csv` as instructed in the code cell. Let $X$ be the data matrix of shape `n_samples × n_dimensions`. Compute an SVD
   $$X = U D V^T$$
   where $U$ has shape `n_samples × n_dimensions`, $D$ is the diagonal matrix of shape `(n_dimensions,n_dimensions)` that has the singular values on the diagonal (see documentation for `np.diag`), and $V$ has shape `n_dimensions × n_dimensions`.
   **Important:** `np.linalg.svd` returns `U, d, Vt` where `Vt` is $V^T$.
   Also extract the **first** right and left singular vectors and store them as 1D arrays in the variables provided.

2. **[3p] Explained variance.** For $N =$ `n_dimensions`, define the explained variance using the first $k$ singular values as
   $$
   \mathrm{EV}(k) = \frac{\sum_{i=1}^k \sigma_i^2}{\sum_{i=1}^N \sigma_i^2}.
   $$
   Compute $\mathrm{EV}(k)$ for $k=1,2,\dots,N$ and store it in `problem1_explained_variance` (length `N`). Then set `problem1_num_components` to the **smallest** $k$ such that $\mathrm{EV}(k) \ge 0.99$.

3. **[3p] Plot + interpretation.** Plot explained variance (x-axis: number of components $k$, y-axis: $\mathrm{EV}(k)$). In the markdown cell below, reason about the shape of the curve for this dataset.

4. **[4p] Low-rank reconstruction + outliers.**
   - Using `problem1_num_components`, construct the best rank-$k$ approximation of $X$ and store it in `problem1_approximation`.
   - Compute the row-wise Euclidean reconstruction error $\|X_i - \hat X_i\|_2$ for each row $i$ and store it in `problem1_reconstruction_error` (shape `(n_samples,)`).
   - Plot the empirical distribution function (EDF) of the reconstruction errors (you may use `makeEDF` / `plotEDF` from `Utils.py`).
   - Choose a threshold `problem1_threshold` so that **exactly 100** samples are flagged as outliers (i.e. have reconstruction error >= the threshold).
   - Store those flagged rows in `problem1_outliers` (shape `(100, n_dimensions)`).


In [222]:
import pandas as pd
import numpy as np


data = pd.read_csv('data/SVD.csv', encoding='latin1')
data.head()

,1.6291346672487743,4.009323395059753,0.9342986235845937,-2.8910437848533648,0.0215747559398553,-0.2070628136731313,1.8239904868036196,-1.5914824056347965,3.127229889440322,4.099373121769439,...,3.6982347097431507,2.694330593337438,3.714049404958557,0.12313092499362191,-0.9135719185458784,-2.925445564727786,-0.8422924399035309,0.38764560140406046,0.6631845626033012,-2.8533868780654843
0,3.518997,8.455528,1.523410,-2.572466,-4.961825,-3.670589,-2.097159,-7.909190,7.151655,0.568081,...,-1.539333,-1.501839,0.928939,-2.573198,-6.878025,-4.456154,-1.499467,-2.874895,-2.141269,-2.321084
1,-0.965495,5.152084,2.400673,3.752781,0.170289,3.449657,4.834497,6.618838,4.320957,3.795531,...,-4.307602,-6.598915,0.892535,-2.319582,3.288029,-2.346942,-5.763055,1.855264,-0.317628,3.235304
2,-2.932740,0.848113,0.140340,-5.503270,3.726912,-4.482047,-8.176017,4.166948,-2.369259,-0.032979,...,7.405467,1.570836,-0.984217,3.469174,-0.482530,1.063566,4.773931,-5.201969,-2.321734,-1.900528
3,-1.400965,0.466787,-0.493385,-3.210639,-5.073728,1.699227,0.635693,-6.737830,3.627136,-4.846562,...,3.656246,-1.368907,0.229606,-3.484510,-3.443440,2.718847,4.323772,-0.821781,5.338389,1.037753
4,2.776900,2.818123,1.824416,1.595839,-1.205149,-3.420631,5.723294,0.411261,5.283886,1.480518,...,-0.248428,-0.154286,1.548275,0.403198,2.573283,-1.558265,-0.274530,0.164079,-2.147975,4.284604


In [223]:
import numpy as np
import pandas as pd

def load_dataset(file_path=None):
    """
    Loads a dataset from a CSV file. Returns a numeric NumPy array.
    """
    if file_path:
        df = pd.read_csv(file_path, encoding='latin1')
    else:
        import seaborn as sns
        df = sns.load_dataset('iris').drop(columns=['species'])
    numeric_df = df.select_dtypes(include=[np.number])
    return numeric_df.to_numpy()

def compute_svd(matrix):
    """Computes full SVD: returns U, d (1D singular values), Vt."""
    U, d, Vt = np.linalg.svd(matrix, full_matrices=True)
    return U, d, Vt


In [224]:
# Part 1: 4 points

# Load data
problem1_data = load_dataset('data/SVD.csv')  # shape: n_samples x n_dimensions
print('dtype:', problem1_data.dtype, '| shape:', problem1_data.shape)

n_samples, n_dimensions = problem1_data.shape

# Compute SVD: X = U D V^T
# np.linalg.svd returns U (n_samples x n_dimensions), d (n_dimensions,), Vt (n_dimensions x n_dimensions)
U_full, d_vals, Vt_full = np.linalg.svd(problem1_data, full_matrices=False)

problem1_U = U_full                          # shape: n_samples x n_dimensions
problem1_D = np.diag(d_vals)                 # shape: n_dimensions x n_dimensions
problem1_V = Vt_full.T                       # shape: n_dimensions x n_dimensions (V, not V^T)

# First right singular vector = first row of Vt, shape (n_dimensions,)
problem1_first_right_singular_vector = Vt_full[0, :].flatten()

# First left singular vector = first column of U, shape (n_samples,)
problem1_first_left_singular_vector = U_full[:, 0].flatten()

print('U shape:', problem1_U.shape)
print('D shape:', problem1_D.shape)
print('V shape:', problem1_V.shape)
print('First right singular vector shape:', problem1_first_right_singular_vector.shape)
print('First left singular vector shape:', problem1_first_left_singular_vector.shape)


In [225]:
import numpy as np

def svd_variance(data):
    """
    Compute variance explained by each singular value from SVD.
    
    Parameters:
        data (np.ndarray): 2D dataset (rows = samples, columns = features)
    
    Returns:
        total_variance (float): Total variance in the dataset
        explained_variance (np.ndarray): Variance explained by each component
        explained_variance_ratio (np.ndarray): Ratio of variance explained
    """
    # Validate input
    if not isinstance(data, np.ndarray):
        raise TypeError("Input must be a NumPy array.")
    if data.ndim != 2:
        raise ValueError("Input must be a 2D array.")
    if data.size == 0:
        raise ValueError("Dataset is empty.")
    
    # Handle NaNs by replacing with column means
    if np.isnan(data).any():
        col_means = np.nanmean(data, axis=0)
        inds = np.where(np.isnan(data))
        data[inds] = np.take(col_means, inds[1])
    
    # Center the data (important for variance calculation)
    data_centered = data - np.mean(data, axis=0)
    
    # Perform SVD
    U, S, Vt = np.linalg.svd(data_centered, full_matrices=False)
    
    # Variance explained by each singular value
    n_samples = data.shape[0]
    explained_variance = (S ** 2) / (n_samples - 1)
    
    # Total variance
    total_variance = explained_variance.sum()
    
    # Ratio of variance explained
    explained_variance_ratio = explained_variance / total_variance
    
    return total_variance, explained_variance, explained_variance_ratio


In [226]:
import numpy as np

def min_singular_values_for_variance(data, threshold=0.99):
    """
    Calculate the minimum number of singular values needed to explain
    at least `threshold` fraction of variance in the dataset.

    Parameters:
        data (np.ndarray): 2D data array (rows = samples, columns = features)
        threshold (float): Fraction of variance to explain (default 0.99)

    Returns:
        int: Minimum number of singular values needed
    """
    # Validate input
    if not isinstance(data, np.ndarray):
        raise TypeError("Data must be a NumPy array.")
    if data.ndim != 2:
        raise ValueError("Data must be a 2D array.")
    if not (0 < threshold <= 1):
        raise ValueError("Threshold must be between 0 and 1.")

    # Center the data (important for variance calculation)
    data_centered = data - np.mean(data, axis=0)

    # Perform Singular Value Decomposition
    U, S, Vt = np.linalg.svd(data_centered, full_matrices=False)

    # Compute variance explained by each singular value
    variance_explained = (S ** 2) / np.sum(S ** 2)

    # Cumulative variance
    cumulative_variance = np.cumsum(variance_explained)

    # Find the smallest numbe1r of singular values to reach threshold
    num_components = np.searchsorted(cumulative_variance, threshold) + 1

    return num_components
    


In [248]:
# Part 2: 3 points

# Singular values (already computed above as d_vals)
sigma_sq = d_vals ** 2
cumulative_ev = np.cumsum(sigma_sq) / np.sum(sigma_sq)

# EV(k) for k = 1, 2, ..., N  (stored as length-N array)
problem1_explained_variance = cumulative_ev  # increasing sequence ending at 1.0

# Smallest k such that EV(k) >= 0.99
problem1_num_components = int(np.searchsorted(cumulative_ev, 0.99) + 1)

print('Explained variance (first 10):', problem1_explained_variance[:10])
print('Num components for 99% variance:', problem1_num_components)


In [228]:
# Part 3: 3 points

import matplotlib.pyplot as plt

N = len(problem1_explained_variance)
plt.figure(figsize=(8, 4))
plt.plot(range(1, N + 1), problem1_explained_variance, 'o-', markersize=3)
plt.axhline(0.99, color='r', linestyle='--', label='99% threshold')
plt.axvline(problem1_num_components, color='g', linestyle='--',
            label=f'k={problem1_num_components}')
plt.xlabel('Number of components k')
plt.ylabel('EV(k)')
plt.title('Explained Variance vs. Number of Components')
plt.legend()
plt.tight_layout()
plt.show()





## Free-text answer (Part 3)



In 2–5 sentences:



- Describe the *shape* of the explained-variance curve.
- Explain why it looks like that for this dataset.



Write your explanation below this line.


the shape of this graphg is in sucha a way because the SVD decreased as components increased.


In [ ]:
# Part 4: 4 points

k = problem1_num_components

# Best rank-k approximation: X_hat = U_k * D_k * Vt_k
U_k   = U_full[:, :k]          # n_samples x k
d_k   = d_vals[:k]             # k,
Vt_k  = Vt_full[:k, :]        # k x n_dimensions

problem1_approximation = (U_k * d_k) @ Vt_k   # shape: n_samples x n_dimensions

# Row-wise Euclidean reconstruction error
problem1_reconstruction_error = np.linalg.norm(
    problem1_data - problem1_approximation, axis=1
)  # shape: (n_samples,)

# Plot EDF of reconstruction errors
sorted_errors = np.sort(problem1_reconstruction_error)
edf = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)
plt.figure(figsize=(8, 4))
plt.plot(sorted_errors, edf)
plt.xlabel('Reconstruction error')
plt.ylabel('EDF')
plt.title('Empirical Distribution Function of Reconstruction Errors')
plt.tight_layout()
plt.show()

# Choose threshold so that exactly 100 samples have error >= threshold
# Sort descending, the 100th largest error is the threshold
problem1_threshold = np.sort(problem1_reconstruction_error)[-100]

# Flag samples with reconstruction error >= threshold
outlier_mask = problem1_reconstruction_error >= problem1_threshold
problem1_outliers = problem1_data[outlier_mask]  # shape: (100, n_dimensions)

print('Threshold:', problem1_threshold)
print('Number of outliers:', len(problem1_outliers))


---
## Exam vB, PROBLEM 2
Maximum Points = 12


In this problem we are interested in **account takeover (ATO) detection** for an online service. You are given the outputs of a classifier that predicts the probability that a login attempt is malicious ($Y=1$). Your goal is to explore how the **decision threshold** affects business cost (as in the thresholding assignment).

A threshold $t \in [0,1]$ is used to convert predicted probabilities $\hat p(x)=P(Y=1\mid x)$ to labels: predict $\hat y=1$ if $\hat p(x)\ge t$, else $\hat y=0$.

The costs associated with the decisions are:

* **True Positive (TP)**: correctly flagging an ATO login costs **80** (extra verification + friction).
* **True Negative (TN)**: allowing a legitimate login has **0** cost.
* **False Positive (FP)**: incorrectly flagging a legitimate login costs **150** (support load + churn risk).
* **False Negative (FN)**: missing an ATO login costs **900** (fraud + remediation).

**The code cells contain more detailed instructions; THE FIRST CODE CELL INITIALIZES YOUR VARIABLES.**

1. **[3p]** Complete the function `problem2_avg_cost` to compute the **average cost per sample** of a model under a certain prediction threshold. Plot the cost as a function of the threshold (using the validation data provided in the first code cell of this problem), from 0 to 1 with step size 0.01.
2. **[2.5p]** Find the threshold that minimizes the cost and calculate the cost at that threshold on the validation data. Also calculate the precision and recall at the optimal threshold, treating **class 1 as the positive class** and **class 0 as the positive class** separately.
3. **[2.5p]** Repeat step 2, but this time find the best threshold to **maximize accuracy** (equivalently, minimize the $0{-}1$ loss). Calculate the difference in cost between the threshold found in part 3 and the one found in part 2.
4. **[4p]** Provide a confidence interval around the optimal cost (with $95\%$ confidence) applied to the test data using **Hoeffding's inequality**, and explain all assumptions you made.

In [261]:

# RUN THIS CELL TO GET THE DATA

# We start by loading the data 

import pandas as pd

PROBLEM2_DF = pd.read_csv('data/fraud.csv')
Y = PROBLEM2_DF['Class'].values
X = PROBLEM2_DF[['V%d' % i for i in range(1,5)]+['Amount']].values

# We will split the data into training, testing and validation sets
from Utils import train_test_validation
PROBLEM2_X_train, PROBLEM2_X_test, PROBLEM2_X_val, PROBLEM2_y_train, PROBLEM2_y_test, PROBLEM2_y_val = train_test_validation(X,Y,shuffle=True,random_state=1)

# From this we will train a logistic regression model with scaling and simple hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(solver='liblinear', max_iter=1000, random_state=1))
])

param_grid = {
    'clf__C': [0.01, 0.1, 1, 10],
    'clf__class_weight': [None, 'balanced']
}

gs = GridSearchCV(pipeline, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
gs.fit(PROBLEM2_X_train, PROBLEM2_y_train)

# use the best pipeline (it supports predict_proba)
lr = gs.best_estimator_

# THE FOLLOWING CODE WILL PRODUCE THE ARRAYS YOU NEED FOR THE PROBLEM
PROBLEM2_y_pred_proba_val = lr.predict_proba(PROBLEM2_X_val)[:,1]
PROBLEM2_y_true_val = PROBLEM2_y_val

PROBLEM2_y_pred_proba_test = lr.predict_proba(PROBLEM2_X_test)[:,1]
PROBLEM2_y_true_test = PROBLEM2_y_test

In [262]:
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# =========================================================
# GIVEN COST STRUCTURE
# =========================================================

# Cost-sensitive classification costs
# TP = True Positive (fraud correctly detected)
# TN = True Negative (legitimate correctly detected)
# FP = False Positive (legitimate flagged as fraud)
# FN = False Negative (fraud missed)
costs = {"TP": 80, "TN": 0, "FP": 150, "FN": 900}

In [245]:
# Part 1: 3 points

import numpy as np
import matplotlib.pyplot as plt

COSTS = {'TP': 80, 'TN': 0, 'FP': 150, 'FN': 900}

def problem2_avg_cost(y_true, y_predict_proba, threshold):
    """
    Average cost per sample given predicted probabilities and a threshold.
    TP=80, TN=0, FP=150, FN=900
    """
    y_true = np.array(y_true)
    y_pred = (np.array(y_predict_proba) >= threshold).astype(int)

    tp = np.sum((y_pred == 1) & (y_true == 1))
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))

    total_cost = COSTS['TP'] * tp + COSTS['TN'] * tn + COSTS['FP'] * fp + COSTS['FN'] * fn
    return total_cost / len(y_true)


# Plot cost vs threshold on validation data
thresholds = np.arange(0, 1.01, 0.01)
costs_val = [problem2_avg_cost(PROBLEM2_y_true_val, PROBLEM2_y_pred_proba_val, t)
             for t in thresholds]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, costs_val)
plt.xlabel('Threshold')
plt.ylabel('Average cost per sample')
plt.title('Cost vs. Decision Threshold (Validation Data)')
plt.tight_layout()
plt.show()


In [246]:
   # Return the average cost per sample (a single float value).

def average_cost_per_sample(TP, TN, FP, FN):
    total_samples = TP + TN + FP + FN
    total_cost = FP + FN  # cost = 1 per misclassification
    return total_cost / total_samples if total_samples else 0.0

# Example usage
print(average_cost_per_sample(80, 0, 150, 900))  # Output: 0.1



0.9292035398230089


In [247]:
import numpy as np

# Example cost function: quadratic (replace with your own)
def cost_function(x):
    return (x - 0.5) ** 2  # Minimum at x = 0.5

def main():
    # Generate values from 0 to 1 with step size 0.01
    x_values = np.arange(0, 1.01, 0.01)  # 1.01 ensures inclusion of 1.0

    # Compute cost for each x
    costs = [cost_function(x) for x in x_values]

    # Display results
    print(f"{'x':>6} | {'Cost':>10}")
    print("-" * 20)
    for x, c in zip(x_values, costs):
        print(f"{x:6.2f} | {c:10.6f}")

if __name__ == "__main__":
    main()


     x |       Cost
--------------------
  0.00 |   0.250000
  0.01 |   0.240100
  0.02 |   0.230400
  0.03 |   0.220900
  0.04 |   0.211600
  0.05 |   0.202500
  0.06 |   0.193600
  0.07 |   0.184900
  0.08 |   0.176400
  0.09 |   0.168100
  0.10 |   0.160000
  0.11 |   0.152100
  0.12 |   0.144400
  0.13 |   0.136900
  0.14 |   0.129600
  0.15 |   0.122500
  0.16 |   0.115600
  0.17 |   0.108900
  0.18 |   0.102400
  0.19 |   0.096100
  0.20 |   0.090000
  0.21 |   0.084100
  0.22 |   0.078400
  0.23 |   0.072900
  0.24 |   0.067600
  0.25 |   0.062500
  0.26 |   0.057600
  0.27 |   0.052900
  0.28 |   0.048400
  0.29 |   0.044100
  0.30 |   0.040000
  0.31 |   0.036100
  0.32 |   0.032400
  0.33 |   0.028900
  0.34 |   0.025600
  0.35 |   0.022500
  0.36 |   0.019600
  0.37 |   0.016900
  0.38 |   0.014400
  0.39 |   0.012100
  0.40 |   0.010000
  0.41 |   0.008100
  0.42 |   0.006400
  0.43 |   0.004900
  0.44 |   0.003600
  0.45 |   0.002500
  0.46 |   0.001600
  0.47 |   0.000900

In [255]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
def train_test_split_table(df):
    """
    Split a data table into training and test sets.

    Returns:
        X_train, X_test, y_train, y_test
    """
    # implement splitting
    # first, decide what are features and what are target 
    X = df[["x1", "x2", "x3"]]   # features
    y = df["fraud"]              # target

    # 2. Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=0,
        stratify=y
    )

    return X_train, X_test, y_train, y_test

def fit_linear_svm(fraud_data):
    """
    Fit a linear SVM classifier.

    Args: data table

    Returns:
        predicted labels of length len(y_test) 
    """
    # define our model
    clf = LinearSVC(
        C=1.0,
        max_iter=10_000,
        random_state=0
    )

    # split the data into train and test
    X_train, X_test, y_train, y_test = train_test_split_table(fraud_data)

    # fit the SVM
    clf.fit(X_train, y_train)

    # predict labels on the test set
    y_pred = clf.predict(X_test)

    return clf

In [256]:
import numpy as np

def find_best_threshold(y_true, y_scores, cost_fn, thresholds=None):
    """
    Finds the threshold that minimizes cost on validation data.

    Parameters:
    - y_true: array-like of shape (n_samples,), true binary labels (0 or 1)
    - y_scores: array-like of shape (n_samples,), predicted probabilities or scores
    - cost_fn: function(y_true, y_pred) -> float, computes cost
    - thresholds: list or np.array of thresholds to evaluate (default: np.linspace(0, 1, 101))

    Returns:
    - best_threshold: threshold value that minimizes cost
    - best_cost: cost at the best threshold
    """
    if thresholds is None:
        thresholds = np.linspace(0, 1, 101)  # 0.00 to 1.00 in steps of 0.01

    best_threshold = None
    best_cost = float("inf")

    for t in thresholds:
        y_pred = (y_scores >= t).astype(int)
        cost = cost_fn(y_true, y_pred)
        if cost < best_cost:
            best_cost = cost
            best_threshold = t

    return best_threshold, best_cost


In [258]:
import numpy as np

def find_optimal_threshold(y_true, y_prob, cost_fp=1.0, cost_fn=1.0):
    """
    Finds the threshold that minimizes cost and returns the threshold and average cost.

    Parameters:
        y_true (array-like): True binary labels (0 or 1).
        y_prob (array-like): Predicted probabilities for the positive class.
        cost_fp (float): Cost of a false positive.
        cost_fn (float): Cost of a false negative.

    Returns:
        best_threshold (float): Threshold that minimizes cost.
        min_avg_cost (float): Average cost at the best threshold.
    """
    # Convert inputs to numpy arrays
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)

    # Validate inputs
    if y_true.shape != y_prob.shape:
        raise ValueError("y_true and y_prob must have the same shape.")
    if not np.all((y_true == 0) | (y_true == 1)):
        raise ValueError("y_true must contain only 0 or 1 values.")
    if not np.all((y_prob >= 0) & (y_prob <= 1)):
        raise ValueError("y_prob must be between 0 and 1.")

    thresholds = np.linspace(0, 1, 101)  # 0.00 to 1.00 in steps of 0.01
    best_threshold = 0.0
    min_avg_cost = float("inf")

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)

        # Calculate false positives and false negatives
        fp = np.sum((y_pred == 1) & (y_true == 0))
        fn = np.sum((y_pred == 0) & (y_true == 1))

        # Total cost
        total_cost = fp * cost_fp + fn * cost_fn
        avg_cost = total_cost / len(y_true)

        if avg_cost < min_avg_cost:
            min_avg_cost = avg_cost
            best_threshold = t

    return best_threshold, min_avg_cost



In [234]:
# Part 2: 2.5 points

from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0, 1.01, 0.01)
costs_val = [problem2_avg_cost(PROBLEM2_y_true_val, PROBLEM2_y_pred_proba_val, t)
             for t in thresholds]

# Threshold minimizing average cost
best_idx = int(np.argmin(costs_val))
problem2_threshold = thresholds[best_idx]   # float
problem2_cost_val  = costs_val[best_idx]    # float

# Predicted labels at optimal threshold
problem2_y_pred_val = (PROBLEM2_y_pred_proba_val >= problem2_threshold).astype(int)

# Precision and recall treating class 1 as positive
problem2_precision_1 = precision_score(PROBLEM2_y_true_val, problem2_y_pred_val, pos_label=1, zero_division=0)
problem2_recall_1    = recall_score(   PROBLEM2_y_true_val, problem2_y_pred_val, pos_label=1, zero_division=0)

# Precision and recall treating class 0 as positive
problem2_precision_0 = precision_score(PROBLEM2_y_true_val, problem2_y_pred_val, pos_label=0, zero_division=0)
problem2_recall_0    = recall_score(   PROBLEM2_y_true_val, problem2_y_pred_val, pos_label=0, zero_division=0)

print(f'Optimal threshold (cost): {problem2_threshold:.2f}')
print(f'Min avg cost (val):       {problem2_cost_val:.4f}')
print(f'Precision (class 1): {problem2_precision_1:.4f}  Recall (class 1): {problem2_recall_1:.4f}')
print(f'Precision (class 0): {problem2_precision_0:.4f}  Recall (class 0): {problem2_recall_0:.4f}')


In [235]:
# Part 3: 2.5 points

thresholds = np.arange(0, 1.01, 0.01)

# Find threshold maximising accuracy (minimising 0-1 loss)
accuracies = [np.mean((PROBLEM2_y_pred_proba_val >= t).astype(int) == PROBLEM2_y_true_val)
              for t in thresholds]
best_acc_idx = int(np.argmax(accuracies))
problem2_threshold_acc = thresholds[best_acc_idx]   # float

# Cost at accuracy-optimal threshold minus cost at cost-optimal threshold
cost_at_acc_threshold  = problem2_avg_cost(PROBLEM2_y_true_val,
                                            PROBLEM2_y_pred_proba_val,
                                            problem2_threshold_acc)
problem2_cost_difference = cost_at_acc_threshold - problem2_cost_val   # float

print(f'Accuracy-optimal threshold: {problem2_threshold_acc:.2f}')
print(f'Cost at acc threshold:      {cost_at_acc_threshold:.4f}')
print(f'Cost difference:            {problem2_cost_difference:.4f}')


In [213]:
def hoeffding_ci(per_obs_costs, mean, n, a, b, delta=0.05):
    """
    Construct a Hoeffding confidence interval for the true expected cost.

    Hoeffding inequality:
        P(|μ̂ − μ| ≥ ε) ≤ 2 exp(−2 n ε² / (b − a)²)

    Solving for ε gives:
        ε = (b − a) * sqrt( log(2 / δ) / (2n) )

    Args:
        per_obs_costs : array of costs per observation
        mean          : empirical mean cost
        n             : number of test observations
        a             : lower bound of cost (0)
        b             : upper bound of cost (500)
        delta         : confidence level (default 0.05 → 95%)

    Returns:
        (lower_bound, upper_bound)
    """

    # Compute Hoeffding radius (confidence half-width)
    epsilon = (b - a) * np.sqrt(np.log(2 / delta) / (2 * n))

    # Confidence interval
    ci_lower = mean - epsilon
    ci_upper = mean + epsilon

    return (ci_lower, ci_upper)

In [259]:
print(hoeffding_ci)

<function hoeffding_ci at 0x0000021F0964F6A0>


In [ ]:
# Part 4: 4 points

# Per-sample costs on test data using cost-optimal threshold
y_pred_test = (PROBLEM2_y_pred_proba_test >= problem2_threshold).astype(int)

def per_sample_costs(y_true, y_pred):
    costs_arr = np.where(
        (y_pred == 1) & (y_true == 1), COSTS['TP'],
        np.where(
            (y_pred == 0) & (y_true == 0), COSTS['TN'],
            np.where(
                (y_pred == 1) & (y_true == 0), COSTS['FP'],
                COSTS['FN']  # FN
            )
        )
    )
    return costs_arr

per_obs = per_sample_costs(PROBLEM2_y_true_test, y_pred_test)
mean_cost_test = per_obs.mean()
n_test = len(per_obs)

# Hoeffding: cost range is [0, 900] (min=TN=0, max=FN=900)
a, b = 0, 900
delta = 0.05
epsilon = (b - a) * np.sqrt(np.log(2 / delta) / (2 * n_test))

problem2_lower_bound = mean_cost_test - epsilon
problem2_upper_bound = mean_cost_test + epsilon

print(f'Mean cost on test data: {mean_cost_test:.4f}')
print(f'Hoeffding epsilon:      {epsilon:.4f}')
print(f'95% CI: [{problem2_lower_bound:.4f}, {problem2_upper_bound:.4f}]')



## Free text answer

Put your explanation for part 4 below this line in this **cell**. Double-click to enter edit mode.

In particular, clearly state:
1. Why Hoeffding's inequality applies (or approximately applies) in this context.
2. What random variables are assumed i.i.d. and what their support is.
3. What bound you used for the per-sample cost range (i.e., $C_{\max} - C_{\min} = ?$ for this problem).

Ans : The Hoeffding confidence interval guarantees that the true expected cost lies within the interval with high probability, providing a reliable but conservative uncertainty estimate. Because it relies only on boundedness and not on the empirical variance, the resulting interval is typically wider than necessary in practice



---
## Exam vB, PROBLEM 3
Maximum Points = 14


A courier company monitors its trucks across four states:

*   **Downtown (D)**
*   **Suburbs (S)**
*   **Countryside (C)**
*   **Maintenance (M)** 

The transition probabilities between states at each time step are given by the matrix:

| Current State | D    | S    | C    | M    |
| ------------- | ---- | ---- | ---- | ---- |
| D             | 0.25 | 0.35 | 0.30 | 0.10 |
| S             | 0.20 | 0.40 | 0.30 | 0.10 |
| C             | 0.15 | 0.35 | 0.40 | 0.10 |
| M             | 0.00 | 0.00 | 0.00 | 1.00 |

1. If a truck starts in the **Suburbs**, what is the probability that it eventually ends up in **Maintenance**? [1p]
2. If a truck starts in **Downtown**, what is the probability that it will be in **Countryside** after five time steps? [2p]
3. Starting from **Downtown**, what is the expected number of steps before entering **Maintenance**? [3p] \
    **Hint**:
    To compute approximatively you could **simulate** but this gives a max score of [1.5p].
    To compute exactly use first-step analysis: 
$$
\text{Expected time from a state} = 1 + \sum_{\text{next states}} \big( \text{transition probability} \times \text{expected time from next state} \big)
$$

4. Is this Markov chain irreducible? Is it aperiodic? [2p]
5. Does this chain have a stationary distribution? If yes, compute it; if not, explain why. [2p]
6. Given that the truck starts in **Countryside** what is the probability that **the last state visited** before reaching **Maintenance** is **Suburbs**? [4p]  
**Hint**: To compute approximatively you could **simulate** but this gives a max score of [2p]. To compute exactly use first-step analysis: Let $f_D, f_S, f_C$ be the probabilities that the last state before hitting Maintenance is Suburbs, starting from Downtown, Suburbs, and Countryside respectively. Write recursive equations by conditioning on the next step. This gives a linear system to solve.


In [73]:
import numpy as np

P = np.array([[0.25, 0.35, 0.30, 0.10],
              [0.20, 0.40, 0.30, 0.10],
              [0.15, 0.35, 0.40, 0.10],
              [0.00, 0.00, 0.00, 1.00]])

downtown    = 0
suburbs     = 1
countryside = 2
maintenance = 3
print(P)


In [74]:
# Start vector for Suburbs (Part 1)
p0_suburbs = np.array([0, 1, 0, 0])
print('Start (Suburbs):', p0_suburbs)


In [304]:
# Part 1 calculation
# M is the unique absorbing state; D, S, C are all transient.
# Every transient state reaches M with probability 1.
prob_to_maintenance = 1.0
print('P(eventually Maintenance | start Suburbs) =', prob_to_maintenance)


In [303]:
# Part 1
problem3_prob_maintenance_from_suburbs = 1.0
print(problem3_prob_maintenance_from_suburbs)


In [302]:
# Start vector for Downtown (Part 2)
p0_downtown = np.array([1, 0, 0, 0])
print('Start (Downtown):', p0_downtown)


In [301]:
# P^5 from Downtown
p5 = p0_downtown @ np.linalg.matrix_power(P, 5)
print('State distribution after 5 steps from Downtown:', p5)
print('P(Countryside after 5 steps):', p5[countryside])


In [300]:
# Part 2
problem3_prob_countryside_after_5_steps = float(p5[countryside])
print(problem3_prob_countryside_after_5_steps)  # ≈ 0.22143


In [299]:
import numpy as np

def expected_steps_to_state(P, target_state):
    """
    Calculate expected number of steps before entering target_state in a Markov chain.
    
    Parameters:
        P (np.ndarray): Transition matrix (n x n), rows sum to 1.
        target_state (int): Index of the target absorbing state (0-based).
    
    Returns:
        np.ndarray: Expected steps from each starting state to reach target_state.
    """
    n = P.shape[0]
    
    # Validate matrix
    if P.shape[0] != P.shape[1]:
        raise ValueError("Transition matrix must be square.")
    if not np.allclose(P.sum(axis=1), 1):
        raise ValueError("Each row of the transition matrix must sum to 1.")
    if not (0 <= target_state < n):
        raise ValueError("Target state index out of range.")
    
    # Make a copy and turn target_state into absorbing
    P_abs = P.copy()
    P_abs[target_state, :] = 0
    P_abs[target_state, target_state] = 1
    
    # Identify transient states
    transient_states = [i for i in range(n) if i != target_state]
    
    # Extract Q (transient-to-transient submatrix)
    Q = P_abs[np.ix_(transient_states, transient_states)]
    
    # Fundamental matrix N = (I - Q)^(-1)
    I = np.eye(len(Q))
    N = np.linalg.inv(I - Q)
    
    # Expected steps from each transient state = sum of rows of N
    t = N.sum(axis=1)
    
    # Map back to original state indices
    expected_steps = np.zeros(n)
    for idx, state in enumerate(transient_states):
        expected_steps[state] = t[idx]
    expected_steps[target_state] = 0  # Already in target
    
    return expected_steps


In [298]:
# Part 3: First-step analysis (exact)
# Let h_i = expected steps to reach Maintenance from state i (transient states: D=0, S=1, C=2)
# h_i = 1 + sum_{j in transient} P[i,j] * h_j
# => (I - Q) h = 1,  where Q = P[0:3, 0:3]
Q = P[:3, :3]
h = np.linalg.solve(np.eye(3) - Q, np.ones(3))
print('Expected steps to Maintenance: D={:.4f}, S={:.4f}, C={:.4f}'.format(*h))


In [297]:
# (helper cell — results used in Part 3 answer below)
pass


In [296]:
# Part 3
problem3_expected_steps_downtown = float(h[downtown])  # = 10.0
print(problem3_expected_steps_downtown)


In [295]:
# Part 4
# Irreducible? NO — M cannot transition back to D, S, or C.
# Aperiodic? YES — D has a self-loop (P[D,D]=0.25), so period=1.
problem3_is_irreducible = False
problem3_is_aperiodic   = True
print('Irreducible:', problem3_is_irreducible, '| Aperiodic:', problem3_is_aperiodic)


In [294]:
# Stationary distribution: solve pi @ P = pi, sum(pi) = 1
# Since M is absorbing, the only stationary dist is pi = [0, 0, 0, 1]
eigenvalues, eigenvectors = np.linalg.eig(P.T)
idx = np.argmin(np.abs(eigenvalues - 1))
stationary = eigenvectors[:, idx].real
stationary = stationary / stationary.sum()
print('Stationary distribution:', stationary)
# Confirms: [0, 0, 0, 1]


In [ ]:
# as u can see the stationary distribution of all the states is 0 excpet "maintenance" therefore the chain does not have a stationary distribution. this is because the markov chain in this question has the "maintenance" vector as zeros and the last cell is 1. this makes the markov chain is reducible and aperiodic, which makes the stationary distribution 

In [293]:
# Part 5
# The chain HAS a stationary distribution. Since M is the unique absorbing state,
# all probability mass is eventually absorbed there.
# pi = [0, 0, 0, 1]
problem3_stationary_distribution = np.array([0.0, 0.0, 0.0, 1.0])
print(problem3_stationary_distribution)


## Free text answer (Part 5)

The chain **does** have a stationary distribution: $\pi = [0, 0, 0, 1]$.

**Why:** Maintenance (M) is an absorbing state — once entered, the chain never leaves it. States D, S, C are all transient (they will be visited finitely many times before absorption). Therefore, the unique stationary distribution places all mass on M: $\pi_M = 1$ and $\pi_D = \pi_S = \pi_C = 0$. The chain is not irreducible (you cannot return from M), but it still has a unique stationary distribution because M is absorbing and every other state is transient.


In [307]:
import numpy as np

def last_before_target_prob(P, target, last_before):
    """
    Compute the probability that the last state visited before reaching `target`
    is `last_before` in a Markov chain.

    Parameters:
        P (np.ndarray): Transition matrix (NxN), rows sum to 1.
        target (int): Index of target absorbing state A.
        last_before (int): Index of state B.
    
    Returns:
        float: Probability starting from each state (array).
    """
    n = P.shape[0]
    if target >= n or last_before >= n:
        raise ValueError("State indices out of range.")
    
    # Make a copy to avoid modifying original
    P_mod = P.copy()
    
    # Make target absorbing
    P_mod[target, :] = 0
    P_mod[target, target] = 1.0
    
    # We want probability that we hit B and then go directly to target
    # without visiting target earlier.
    
    # Step 1: Remove direct transitions to target except from B
    for i in range(n):
        if i != last_before and i != target:
            P_mod[i, target] = 0
            # Renormalize row to sum to 1
            row_sum = P_mod[i].sum()
            if row_sum > 0:
                P_mod[i] /= row_sum
    
    # Step 2: Solve for probability of reaching B before target
    # Let x[i] = probability starting from i that we reach B before target
    Q = np.delete(np.delete(P_mod, target, axis=0), target, axis=1)
    b_index = last_before if last_before < target else last_before - 1
    
    # Build RHS vector: 1 if state is B, else 0
    rhs = np.zeros(n - 1)
    rhs[b_index] = 1.0
    
    # Solve (I - Q) x = rhs
    I = np.eye(Q.shape[0])
    x = np.linalg.solve(I - Q, rhs)
    
    # Step 3: Multiply by probability of going from B to target
    prob_B_to_target = P[last_before, target]
    
    # Insert target state's probability as 0
    result = np.insert(x * prob_B_to_target, target, 0.0)
    return result


P = np.array([[0.25, 0.35, 0.3 , 0.1 ],
        [0.2 , 0.4 , 0.3 , 0.1 ],
        [0.15, 0.35, 0.4 , 0.1 ],
        [0.  , 0.  , 0.  , 1.  ]])

target_state = 3  # A
last_before_state = 1  # B

probs = last_before_target_prob(P, target_state, last_before_state)
print("Probability from each state:", probs)

Probability from each state: [1. 1. 1. 0.]


In [291]:
# Part 6: exact solution via first-step analysis
# Let f_i = P(last state before M is Suburbs | start at i), for i in {D, S, C}
#
# If we go to M directly from state i, the last state is i itself.
# f_D = P[D,D]*f_D + P[D,S]*f_S + P[D,C]*f_C + P[D,M]*0
# f_S = P[S,D]*f_D + P[S,S]*f_S + P[S,C]*f_C + P[S,M]*1
# f_C = P[C,D]*f_D + P[C,S]*f_S + P[C,C]*f_C + P[C,M]*0
#
# Rearranging (I - Q_T) f = r  where r = [0, P[S,M], 0] = [0, 0.1, 0]

A = np.eye(3) - P[:3, :3]   # (I - Q)
r = np.array([0.0, P[suburbs, maintenance], 0.0])  # [0, 0.1, 0]
f = np.linalg.solve(A, r)
print('f_D={:.4f}, f_S={:.4f}, f_C={:.4f}'.format(*f))

problem3_prob_last_suburbs_from_countryside = float(f[countryside])  # ≈ 0.3684
print('Answer:', problem3_prob_last_suburbs_from_countryside)


In [308]:
# See Part 6 cell above for the correct solution.
